In [ ]:
def svm_recommender():
    import pandas as pd
    import numpy as np
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.model_selection import train_test_split
    from sklearn.svm import SVC
    from sklearn.preprocessing import StandardScaler
    from scipy.sparse import vstack
    import gc

    # 1.load and vectorise posts
    def post_vectors():
        df = pd.read_csv('reddit_data.csv')
        df['timestamp'] = pd.to_datetime(df['created_utc'], unit='s')
        df = df.sort_values(by='timestamp')
        df = df[df['title'].notna() & (df['title'].str.strip() != '')]
        df['selftext'] = df['selftext'].fillna('')
#         split into train adn test data 
        split_idx = int(len(df) * 0.8)
        train_df = df.iloc[:split_idx]
        test_df = df.iloc[split_idx:]
        
        combined_text = df['title'] + ' ' + df['selftext']
        vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
        X = vectorizer.fit_transform(combined_text)
        # dicitonary of each posts mapped to its vector 
        submission_id_to_vector = {
            sid: X[i] for i, sid in enumerate(df["submission_id"])
        }
        return df, train_df, test_df, submission_id_to_vector

    # 2. Get upvoted posts in test set (for evaluation later)
    def get_upvoted_posts_by_user(votes, test_df, subreddit="r/Showerthoughts"):
        filtered = votes[
            (votes["SUBREDDIT"] == subreddit) &
            (votes["VOTE"] == "upvote") &
            (votes["SUBMISSION_ID"].isin(set(test_df["submission_id"])))
        ]
        return filtered.groupby("USERNAME")["SUBMISSION_ID"].apply(set).to_dict()

    # 3. Get user votes
    def get_user_votes(votes, subreddit: str = "r/Showerthoughts") -> dict:
        #filter so its only votse from Showerthoguhts
        filtered = votes[votes["SUBREDDIT"] == subreddit]
        # seperate upovotes and downvotes
        upvotes = filtered[filtered["VOTE"] == "upvote"]
        downvotes = filtered[filtered["VOTE"] == "downvote"]

        user_upvoted = upvotes.groupby("USERNAME")["SUBMISSION_ID"].apply(set)
        user_downvoted = downvotes.groupby("USERNAME")["SUBMISSION_ID"].apply(set)

        # sort by desscending order 
        user_vote_counts = (
            user_upvoted.apply(len).add(user_downvoted.apply(len), fill_value=0)
            .sort_values(ascending=False)
        )

        top_users = user_vote_counts.head(20).index

        # user_votes dictionary
        user_votes = {
            user: {
                "positive": user_upvoted.get(user, set()),
                "negative": user_downvoted.get(user, set())
            }
            for user in top_users
        }
        return user_votes


    # 4. Train models
    def train_models(user_votes, submission_id_to_vector, train_df):
        user_models = {}
        train_post_ids = set(train_df["submission_id"])
        for user, prefs in user_votes.items():
            pos_vecs = [
                submission_id_to_vector[submission_id]
                for submission_id in prefs['positive']
                if submission_id in submission_id_to_vector and submission_id in train_post_ids
            ]
            neg_vecs = [
                submission_id_to_vector[submission_id]
                for submission_id in prefs['negative']
                if submission_id in submission_id_to_vector and submission_id in train_post_ids
            ]
            if len(pos_vecs) < 3 or len(neg_vecs) < 3:
                continue
            X_train = vstack(pos_vecs + neg_vecs)
            y_train = [1] * len(pos_vecs) + [0] * len(neg_vecs)

            scaler = StandardScaler(with_mean=False)
            X_scaled = scaler.fit_transform(X_train)

            svm = SVC(kernel='linear', class_weight='balanced')
            svm.fit(X_scaled, y_train)
            user_models[user] = {"model": svm, "scaler": scaler}

            del X_train, X_scaled, y_train, svm, scaler
            gc.collect()
        return user_models

    # 5. Recommendation for a user
    def recommend_for_user(user, user_models, submission_id_to_vector, test_df, top_n):
        rec_post_ids = test_df['submission_id'].tolist()
        rec_vec = [submission_id_to_vector[post_id] for post_id in rec_post_ids if post_id in submission_id_to_vector]
        scaler = user_models[user]['scaler']
        model = user_models[user]['model']
        X_scaled = scaler.transform(vstack(rec_vec))
        scores = model.decision_function(X_scaled)
        scored_posts = sorted(zip(rec_post_ids, scores), key=lambda x: x[1], reverse=True)
        
        return [post_id for post_id, _ in scored_posts[:top_n]]
    
    # 6. ndcg calculation
    def ndcg_at_k(recommended, actual_upvotes, k):
        if not actual_upvotes:
            return 0.0
        # create relevance scores 
        relevance = [1 if item in actual_upvotes else 0 for item in recommended[:k]]
        # dcg
        dcg = sum((2 ** rel - 1) / np.log2(idx + 2) 
                 for idx, rel in enumerate(relevance))
        #  idcg
        ideal_relevance = sorted([1] * min(len(actual_upvotes), k), reverse=True)
        idcg = sum((2 ** rel - 1) / np.log2(idx + 2) 
                  for idx, rel in enumerate(ideal_relevance))

        return dcg / idcg if idcg > 0 else 0
#     7. evalutate
    def evaluate(user_votes, actual_votes_positive, user_models, submission_id_to_vector, test_df, k):
        total_ndcg, recommended_items = 0, set()
        all_items = set(test_df['submission_id'])
        user_count = 0
        for user in user_votes:
            if user not in user_models:
                ndcg = 0
            else:
                actual_upvotes = actual_votes_positive.get(user, set())
                recs = recommend_for_user(user, user_models, submission_id_to_vector, test_df, k)
                ndcg = ndcg_at_k(recs, actual_upvotes, k)
                total_ndcg += ndcg
        
            recommended_items.update(recs)
            user_count += 1
            print("ndcg for user", user, ndcg)
        return {
            "ndcg@k": total_ndcg / user_count if user_count else 0,
            "users_evaluated": user_count
        }

    # Pipeline execution ---
    df, train_df, test_df, submission_id_to_vector = post_vectors()
    user_votes = get_user_votes(votes)
    actual_votes_positive = get_upvoted_posts_by_user(votes, test_df)
    user_models = train_models(user_votes, submission_id_to_vector, train_df)

    results = evaluate(user_votes, actual_votes_positive, user_models, submission_id_to_vector, test_df, k=200)

    print("Evaluation Results:")
    print(f"NDCG@200: {results['ndcg@k']:.4f}")
    print(f"Users evaluated: {results['users_evaluated']}")
